In [4]:
import torch.nn as nn
import torch
from torch.utils.data import TensorDataset, DataLoader, random_split
import pandas as pd
from collections import Counter
import random
import time
import math
import spacy

In [4]:
df = pd.read_csv("proj_dataset.csv")
df = df.dropna()

In [5]:
all_tokens = []
for sentence in (df['inputReviews'].tolist() + df['outputReviews'].tolist()):
    for word in sentence.lower().split():
        all_tokens.append(word)

In [6]:
print(len(all_tokens))

378742


In [7]:
vocab = {'<pad>': 0, '<sos>': 1, '<eos>': 2, '<unk>': 3}
for token, _ in Counter(all_tokens).most_common():
    vocab[token] = len(vocab)

In [8]:
def encode(sentence):
    return [vocab.get(tok, vocab['<unk>']) for tok in sentence.lower().split()]

def encode_with_special(sentence):
    tokens = [vocab['<sos>']] + encode(sentence) + [vocab['<eos>']]
    return torch.tensor(tokens, dtype=torch.long)

In [9]:
inp_tensors = [encode_with_special(s) for s in df['inputReviews']]
out_tensors = [encode_with_special(s) for s in df['outputReviews']]

In [10]:
inp_padded = nn.utils.rnn.pad_sequence(inp_tensors, batch_first=True, padding_value=0)
out_padded = nn.utils.rnn.pad_sequence(out_tensors, batch_first=True, padding_value=0)

In [82]:
print(inp_padded.shape)

torch.Size([10797, 274])


In [11]:
dataset = TensorDataset(inp_padded, out_padded)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

train_size = int(0.7 * len(dataset))
val_size   = int(0.15 * len(dataset))
test_size  = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=16, shuffle=False)